# 📓 Notebook 11 — Multi-Agent Systems---**Phase 4 · Agent Architecture**> One agent is limited. Multiple specialized agents, coordinated by a manager, can tackle complex workflows.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Manager-Worker pattern | One agent delegates, others execute || Agent specialization | Each agent has a focused role || Fan-out / Fan-in | Parallel work + results aggregation || Delegation & routing | Matching tasks to the right agent |### 🏗️ Build: Research Team Simulator

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---## 2. Multi-Agent Architecture### The Manager-Worker Pattern```         ┌─────────────┐         │   Manager    │  ← Decomposes task, delegates, synthesizes         └──────┬───────┘          ┌─────┼─────┐          ↓     ↓     ↓       ┌────┐┌────┐┌────┐       │ W1 ││ W2 ││ W3 │  ← Specialized worker agents       └────┘└────┘└────┘```### Fan-Out / Fan-In Pattern```        Task → [Split into N sub-tasks]                ↓ (Fan-Out)    ┌───────┬───────┬───────┐    │ Sub-1 │ Sub-2 │ Sub-3 │  (Parallel execution)    └───┬───┘───┬───┘───┬───┘        └───────┼───────┘                ↓ (Fan-In)        [Aggregate results]                ↓           Final Output```> **Interview Tip:** *"What are the coordination patterns for multi-agent systems?"*  > 1. Manager-Worker (hierarchical) 2. Peer-to-peer (agents communicate directly) 3. Pipeline (chain of agents) 4. Fan-out/Fan-in (parallel + aggregate)

In [ ]:
class Agent:    """Base agent with a specific role and expertise."""        def __init__(self, name, role, system_prompt, model=None):        self.name = name        self.role = role        self.system_prompt = system_prompt        self.model = model or DEFAULT_MODEL        def run(self, task, context=""):        """Execute a task."""        messages = [            {"role": "system", "content": self.system_prompt},            {"role": "user", "content": f"Task: {task}\n\nContext: {context}" if context else f"Task: {task}"}        ]                response = litellm.completion(            model=self.model, messages=messages,            temperature=0.4, max_tokens=600,        )        return response.choices[0].message.content        def __repr__(self):        return f"Agent({self.name}, role={self.role})"# Define specialized agentsplanner_agent = Agent(    name="Planner",    role="research_planner",    system_prompt="You are a research planner. Break down research topics into 3-4 specific, researchable questions. Return as a JSON array of strings under key 'questions'.")researcher_agent = Agent(    name="Researcher",    role="researcher",    system_prompt="You are a thorough researcher. Given a question, provide detailed, factual information with specific data points. Be comprehensive but concise.")writer_agent = Agent(    name="Writer",    role="writer",    system_prompt="You are a professional technical writer. Given research notes, write a well-structured, clear report section. Use headers, bullet points, and highlight key findings.")reviewer_agent = Agent(    name="Reviewer",    role="reviewer",    system_prompt="You are a critical reviewer. Review the given report for accuracy, completeness, clarity, and actionability. Provide specific feedback and a quality score from 1-10.")print("✅ 4 Agents defined:")for agent in [planner_agent, researcher_agent, writer_agent, reviewer_agent]:    print(f"   {agent}")

In [ ]:
class ResearchTeam:    """Multi-agent research team with Manager-Worker + Fan-out/Fan-in patterns."""        def __init__(self):        self.planner = planner_agent        self.researcher = researcher_agent        self.writer = writer_agent        self.reviewer = reviewer_agent        self.log = []        def _log(self, agent_name, action, content_preview):        self.log.append({"agent": agent_name, "action": action, "preview": content_preview[:80]})        print(f"  [{agent_name}] {action}: {content_preview[:60]}...")        def research(self, topic, verbose=True):        """Run the full research pipeline."""        if verbose:            print(f"🚀 Research Team — Topic: {topic}")            print("=" * 60)                # Phase 1: Planning (Planner Agent)        if verbose: print("\n📋 Phase 1: Planning")        plan_output = self.planner.run(topic)        self._log("Planner", "planned", plan_output)                try:            plan_data = json.loads(plan_output)            questions = plan_data.get("questions", [topic])        except:            questions = [topic]                if verbose:            for i, q in enumerate(questions, 1):                print(f"   Q{i}: {q}")                # Phase 2: Research (Fan-Out — one researcher per question)        if verbose: print(f"\n🔍 Phase 2: Research (fan-out: {len(questions)} tasks)")        research_results = []        for q in questions:            result = self.researcher.run(q)            research_results.append({"question": q, "findings": result})            self._log("Researcher", f"researched", result)                # Phase 3: Writing (Fan-In — combine research into report)        if verbose: print("\n✍️ Phase 3: Writing (fan-in)")        all_research = "\n\n".join([            f"## {r['question']}\n{r['findings']}" for r in research_results        ])        report = self.writer.run(            f"Write a comprehensive report on: {topic}",            context=all_research        )        self._log("Writer", "wrote report", report)                # Phase 4: Review        if verbose: print("\n🔎 Phase 4: Review")        review = self.reviewer.run(            f"Review this research report on '{topic}'",            context=report        )        self._log("Reviewer", "reviewed", review)                if verbose:            print(f"\n{'=' * 60}")            print(f"📄 FINAL REPORT")            print(f"{'=' * 60}")            print(report[:500])            print(f"\n{'─' * 60}")            print(f"📝 REVIEW")            print(review[:300])                return {            "topic": topic,            "questions": questions,            "research": research_results,            "report": report,            "review": review,            "steps": len(self.log),        }# Run the research teamteam = ResearchTeam()result = team.research("The impact of Large Language Models on software engineering productivity")

In [ ]:
# Review the agent logprint("\n📊 Agent Activity Log")print("=" * 60)for i, entry in enumerate(team.log, 1):    print(f"  {i}. [{entry['agent']}] {entry['action']}: {entry['preview']}")print(f"\n  Total agent interactions: {len(team.log)}")

---## 3. Dynamic Routing — Matching Tasks to Agents

In [ ]:
class AgentRouter:    """Routes tasks to the most appropriate agent."""        def __init__(self, agents, model=None):        self.agents = {a.role: a for a in agents}        self.model = model or DEFAULT_MODEL        def route(self, task):        """Determine which agent should handle this task."""        agent_descriptions = "\n".join([            f"- {a.role}: {a.system_prompt[:100]}" for a in self.agents.values()        ])                response = litellm.completion(            model=self.model,            messages=[{                "role": "user",                "content": f"Which agent should handle this task? Available agents:\n{agent_descriptions}\n\nTask: {task}\n\nRespond with ONLY the agent role name."            }],            temperature=0, max_tokens=50,        )                role = response.choices[0].message.content.strip().lower()        return self.agents.get(role, list(self.agents.values())[0])router = AgentRouter([planner_agent, researcher_agent, writer_agent, reviewer_agent])# Test routingtasks = [    "Break down the topic of quantum computing",    "Find information about GPU prices in 2024",    "Write a summary of the research findings",    "Check if this report has any factual errors",]print("🔀 Agent Routing Demo")print("=" * 50)for task in tasks:    agent = router.route(task)    print(f"  Task: {task[:40]}... → Agent: {agent.name}")

---## 📝 Interview Quiz — Multi-Agent Systems### Q1: Manager-Worker vs Peer-to-Peer — compare the two patterns.<details><summary>💡 Answer</summary>**Manager-Worker (Hierarchical):**- One manager delegates, coordinates, and aggregates- Clear chain of command, easy to debug- Single point of failure (manager)- Best for: structured workflows, report generation**Peer-to-Peer (Decentralized):**- Agents communicate directly with each other- No single point of failure- Harder to coordinate and debug- Best for: debate, negotiation, consensus-building**In practice:** Most production systems use Manager-Worker for predictability.</details>### Q2: What is the fan-out/fan-in pattern? Give a real-world example.<details><summary>💡 Answer</summary>**Fan-out:** Split one task into N parallel sub-tasks**Fan-in:** Aggregate N results back into one output**Example — Research Report:**1. Manager receives: "Analyze the EV market"2. Fan-out: Sends 4 sub-queries to researcher agents (market size, key players, technology trends, regulatory landscape)3. Parallel execution: All 4 researchers work simultaneously4. Fan-in: Writer agent aggregates all findings into one report**MapReduce is the same pattern!** Map = fan-out, Reduce = fan-in.</details>### Q3: How do you prevent conflicting outputs from multiple agents?<details><summary>💡 Answer</summary>1. **Clear role boundaries** — Each agent has a specific scope (no overlap)2. **Single source of truth** — One agent has final authority for each decision3. **Structured output** — Use JSON schemas to enforce consistent format4. **Reviewer agent** — Dedicated agent that checks for contradictions5. **Voting/consensus** — Multiple agents answer, take majority vote6. **Pipeline order** — Later agents refine earlier outputs (not override)</details>### Q4: What are the costs of multi-agent systems?<details><summary>💡 Answer</summary>- **Latency:** More agents = more LLM calls = slower overall- **Token cost:** Each agent has its own system prompt + context- **Complexity:** Harder to debug than single-agent- **Coordination overhead:** Manager routing, message passing- **Error propagation:** One agent's mistake can cascade**When worth it:** When the task genuinely benefits from specialization and the quality improvement justifies the cost.**Rule of thumb:** Start with a single well-prompted agent. Only add agents when a single agent's output quality is insufficient.</details>

---## ✅ Notebook 11 Summary| Concept | Key Takeaway ||---------|-------------|| Manager-Worker | Hierarchical delegation, clear control flow || Fan-out/Fan-in | Parallel execution + result aggregation || Agent specialization | Each agent has focused role and system prompt || Dynamic routing | LLM-based task-to-agent matching || Trade-offs | More agents = better quality but higher cost/latency || When to use | Only when single-agent quality is insufficient |### ➡️ Next: [Notebook 12 — Reflection & Self-Improvement](./12_reflection.ipynb)